In [19]:
import os
import glob
import multiprocessing as mp
import matplotlib
#matplotlib.use('Agg') 
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import pandas as pd
import gc



def plot_three(name_lc_raw, name_lc_outliers, name_lc_cbv):
    ''' function that  take the lc information for plotting '''
    #getting information for raw light curve
    lc_raw = pd.read_csv(name_lc_raw, sep='\s+',
                    names = ['JD', 'mag', 'err'])
    lc_raw = lc_raw.apply(pd.to_numeric, errors='coerce').dropna() #---> applying nan

    #getting information for raw light curve with outliers
    lc_outliers = pd.read_csv(name_lc_outliers, sep=',',
                    names = ['JD', 'mag', 'err'])
    lc_outliers = lc_outliers.apply(pd.to_numeric, errors='coerce').dropna()

    #getting information for raw light curve with median after cbv
    lc_cbv = pd.read_csv(name_lc_cbv, sep='\s+',
                     names=['JD', 'mag_clean', 'mag_after_cbv', 'err'])
    lc_cbv = lc_cbv.apply(pd.to_numeric, errors='coerce').dropna()
    
    folder_name = os.path.basename(os.path.dirname(name_lc_raw))
    base_name = os.path.basename(name_lc_raw)
    without_ext = os.path.splitext(base_name)[0]
    parts = without_ext.split('_')
    name_obj = parts[0]
    name_pdf = f"TESS light curve {name_obj}  {folder_name}.pdf" #name of the pdf
    sector = '_'.join(parts[1:])
    

    

        # using seaborn for aestetic reason
    sns.set_theme(style="ticks", context="paper", palette="viridis")
    sns.set_style({"xtick.direction": "in", "ytick.direction": "in"})

        # Creating a plot 
    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(10, 9), sharex=True,
                                        gridspec_kw={'height_ratios': [1, 1, 1]})
        # ==================== PANEL 1: RAW ====================
    if not lc_raw.empty:
        ax1.errorbar(lc_raw['JD'], lc_raw['mag'], yerr=lc_raw['err'],
                     fmt='none', ecolor='green', alpha=0.1,
                     capsize=0, zorder=1, label='Error (mag)')
        sns.scatterplot(data=lc_raw, x='JD', y='mag', hue='mag',
                        palette='viridis', s=5, legend=False, ax=ax1)
        ax1.invert_yaxis()
        ax1.set_ylabel('Magnitude')
        ax1.set_title(f'TESS {name_obj} {folder_name} Raw light curve')
        ax1.grid(True, linestyle='--', alpha=0.3)
        ax1.legend(loc='lower left', framealpha=0.5)
    else:
        ax1.text(0.5, 0.5, 'No valid data', transform=ax1.transAxes, ha='center', va='center')
        ax1.set_title(f'TESS {name_obj} {folder_name} Raw light curve')
    sns.despine(top=True, right=True, ax=ax1)

    # ==================== PANEL 2: OUTLIER CLEANED ====================
    if not lc_outliers.empty:
        ax2.errorbar(lc_outliers['JD'], lc_outliers['mag'], yerr=lc_outliers['err'],
                     fmt='none', ecolor='green', alpha=0.1,
                     capsize=0, zorder=1, label='Error (mag)')
        sns.scatterplot(data=lc_outliers, x='JD', y='mag', hue='mag',
                        palette='viridis', s=5, legend=False, ax=ax2)
        ax2.invert_yaxis()
        ax2.set_ylabel('Magnitude')
        ax2.set_title(f'TESS {name_obj} {folder_name} Outlier cleaned')
        ax2.grid(True, linestyle='--', alpha=0.3)
        ax2.legend(loc='lower left', framealpha=0.5)
    else:
        ax2.text(0.5, 0.5, 'No valid data', transform=ax2.transAxes, ha='center', va='center')
        #ax2.set_title(f'TESS {name_obj} {folder_name} Outlier cleaned')
    sns.despine(top=True, right=True, ax=ax2)

    # ==================== PANEL 3: MEDIAN AFTER CBV ====================
    if not lc_cbv.empty:
        ax3.errorbar(lc_cbv['JD'], lc_cbv['mag_after_cbv'], yerr=lc_cbv['err'],
                     fmt='none', ecolor='green', alpha=0.1,
                     capsize=0, zorder=1, label='Error (mag)')
        sns.scatterplot(data=lc_cbv, x='JD', y='mag_after_cbv', hue='mag_after_cbv',
                        palette='viridis', s=5, legend=False, ax=ax3)
        ax3.invert_yaxis()
        ax3.set_ylabel('Magnitude')
        ax3.set_title(f'TESS {name_obj} {folder_name} Median after CBV (detrended)')
        ax3.grid(True, linestyle='--', alpha=0.3)
        ax3.legend(loc='lower left', framealpha=0.5)
    else:
        ax3.text(0.5, 0.5, 'No valid data', transform=ax3.transAxes, ha='center', va='center')
        ax3.set_title(f'TESS {name_obj} {folder_name} (detrended)')
    sns.despine(top=True, right=True, ax=ax3)

    # Etiqueta común para el eje X (solo en el último panel)
    ax3.set_xlabel('Julian Date')

    plt.tight_layout()
    plt.savefig(name_pdf,dpi =300)
    plt.show()
    plt.close()


def process_all_triplets(raw_root, outlier_root, median_root,
                         outlier_suffix='_cleaned',
                         median_subfolder_prefix='_lc_median_after_cbv_detrended_'):
    """
    Walks raw_root and for each .lc:
        - outlier: same relative path + suffix '_cleaned'
        - median: subfolder name = median_subfolder_prefix + original_subfolder_name
    """
    for root, dirs, files in os.walk(raw_root):
        for file in files:
            if not file.endswith('.lc'):
                continue

            raw_path = os.path.join(root, file)
            base = os.path.splitext(file)[0]          # e.g. '41259805_sector01_4_2'

            # Relative path from raw_root (e.g. 'ACV' or subsubfolders)
            rel_path = os.path.relpath(root, raw_root)
            if rel_path == '.':
                rel_path = ''

            # --- Outlier: same rel_path, add suffix ---
            outlier_name = base + outlier_suffix + '.lc'
            outlier_path = os.path.join(outlier_root, rel_path, outlier_name)
            if not os.path.exists(outlier_path):
                print(f"⚠️ Outlier not found: {outlier_path}")
                continue

            # --- Median: transform the subfolder name ---
            # Build the median subfolder: prefix + original subfolder
            # e.g. rel_path = 'ACV' → median_subfolder = '_lc_median_after_cbv_detrended_ACV'
            if rel_path:
                median_subfolder = median_subfolder_prefix + rel_path
            else:
                median_subfolder = ''   # if raw files are directly in root (no subfolder)
            median_path = os.path.join(median_root, median_subfolder, file)
            if not os.path.exists(median_path):
                print(f"⚠️ Median not found for {raw_path}")
                print(f"   Expected: {median_path}")
                continue

            # --- Plot ---
            print(f"✓ Processing {base} (median found at {median_path})")
            plot_three(raw_path, outlier_path, median_path)

# ------------------------------------------------------------
# 3. Execution
# ------------------------------------------------------------
if __name__ == "__main__":
    raw_folder = "_TESS_lightcurves_raw"
    outlier_folder = "_TESS_lightcurves_outliercleaned"
    median_folder = "_TESS_lightcurves_median_after_detrended"

    # You can change median_subfolder_prefix if needed
    process_all_triplets(raw_folder, outlier_folder, median_folder,
                         outlier_suffix='_cleaned',
                         median_subfolder_prefix='_lc_median_after_cbv_detrended_')

✓ Processing 41259805_sector01_4_2 (median found at _TESS_lightcurves_median_after_detrended/_lc_median_after_cbv_detrended_ACV/41259805_sector01_4_2.lc)


/tmp/ipykernel_157131/2175749451.py:105: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


✓ Processing 92705248_sector01_1_1 (median found at _TESS_lightcurves_median_after_detrended/_lc_median_after_cbv_detrended_ACV/92705248_sector01_1_1.lc)


/tmp/ipykernel_157131/2175749451.py:105: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


✓ Processing 238869272_sector01_3_2 (median found at _TESS_lightcurves_median_after_detrended/_lc_median_after_cbv_detrended_ACV/238869272_sector01_3_2.lc)


/tmp/ipykernel_157131/2175749451.py:105: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


✓ Processing 270304671_sector01_1_3 (median found at _TESS_lightcurves_median_after_detrended/_lc_median_after_cbv_detrended_ACV/270304671_sector01_1_3.lc)


/tmp/ipykernel_157131/2175749451.py:105: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
